# Malaysia Financial Analytics Platform  
## Data Management Pipeline Using Apache Hive and Python

This notebook forms the data management layer of the Malaysia Financial Performance Analytics Platform. The project uses open-source World Bank Malaysia financial and economic indicators to develop a structured analytical dataset for financial analysis, dashboard development and documentary-style data visualisation.

The data management process begins with raw World Bank data that was uploaded into HDFS and processed using Apache Hive. The selected indicators exported from Hive are then cleaned, reshaped, validated and prepared in Python. The final output of this notebook is a clean analytical dataset that will be used in R for visual analytics, heatmaps, correlation analysis, dashboard development and storyboard presentation.

## Project Information

| Item | Description |
|------|-------------|
| Project | Malaysia Financial Performance Analytics Platform |
| Course | Data Management & Data Visualisation |
| Dataset | World Bank Malaysia Financial Indicators |
| Data Source | World Bank Open Data |
| Data Engineering | Apache Hive (HDFS) |
| Data Management | Python |
| Visual Analytics | R |
| Dashboard | R Shiny |
| Output | Analytical Dataset, Interactive Dashboard, Documentary Storyboard |

## Business Problem

Malaysia's financial performance is closely linked to the daily financial well-being of ordinary citizens. Inflation affects purchasing power, unemployment affects income security, interest rates affect borrowing and saving decisions, household consumption reflects spending behaviour, savings reflect financial resilience, while domestic credit and non-performing loans indicate the ability and health of the financial system to support households and businesses.

This project develops a structured data management pipeline to transform raw World Bank Malaysia data into a clean analytical dataset for visual analytics, executive reporting and dashboard development.

## Data Source and Hive Output

The original dataset was obtained from the World Bank open data source. The raw Malaysia dataset was uploaded into HDFS and processed using Apache Hive. Hive was used to create an external table and filter selected financial and economic indicators relevant to Malaysia's finance and banking sector.

The selected Hive output was exported as a CSV file and used as the input for this Python data management pipeline.

## Data Pipeline Overview

The Malaysia Financial Performance Analytics Platform follows a structured Extract–Transform–Load (ETL) workflow to transform raw World Bank economic and financial data into a clean analytical dataset for decision support and visual analytics.

The project consists of the following stages:

1. Acquisition of the World Bank Malaysia raw dataset.
2. Storage and validation of the dataset in HDFS.
3. Data organisation and verification using Apache Hive.
4. Loading the raw dataset into Python.
5. Data selection, transformation and cleaning.
6. Data quality assessment and validation.
7. Feature engineering and analytical dataset creation.
8. Export of the final analytical dataset.
9. Visual analytics and interactive dashboard development using R and Shiny.
10. Documentary storyboard and executive presentation.

The final indicator framework is organised into three themes:

1. Economic Well-being
2. Household Financial Behaviour
3. Financial System Support

## PHASE 1 – DATA UNDERSTANDING

## Load Raw Dataset

The original World Bank Malaysia dataset is loaded into Python as the starting point of the data management pipeline. The raw data contain multiple economic and financial indicators across several years and require filtering and transformation before analysis.

Apache Hive was used as part of the data engineering workflow to validate and organise the data. This notebook reconstructs the analytical pipeline from the raw dataset to produce a clean and reusable dataset for visual analytics.

In [1]:
# ============================================================
# 1. Load Required Libraries
# ============================================================

import pandas as pd
import numpy as np
import os
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# 2. Folder Setup
# ============================================================

BASE_DIR = Path("..")

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_FINAL = BASE_DIR / "data" / "final"
DATA_METADATA = BASE_DIR / "data" / "metadata"

for folder in [DATA_RAW, DATA_PROCESSED, DATA_FINAL, DATA_METADATA]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders checked and ready.")

Project folders checked and ready.


In [3]:
# ============================================================
# 3. Load Raw Dataset
# ============================================================

RAW_FILE = DATA_RAW / "worldbank_malaysia_raw.csv"

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw dataset not found at: {RAW_FILE}")

raw_df = pd.read_csv(RAW_FILE)

print("Raw dataset loaded successfully.")
print(f"Rows: {raw_df.shape[0]}")
print(f"Columns: {raw_df.shape[1]}")

Raw dataset loaded successfully.
Rows: 1486
Columns: 70


## Inspect Dataset Structure

The structure of the raw dataset is inspected to identify metadata fields and yearly observations before any transformation is performed.

In [4]:
# ============================================================
# 4. Inspect Dataset Structure
# ============================================================

metadata_columns = raw_df.columns[:4].tolist()
year_columns = [col for col in raw_df.columns if col.isdigit()]

print("Metadata columns:")
print(metadata_columns)

print(f"\nYear coverage : {min(year_columns)} - {max(year_columns)}")
print(f"Number of yearly columns : {len(year_columns)}")

Metadata columns:
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']

Year coverage : 1960 - 2025
Number of yearly columns : 66


## Initial Data Quality Assessment

The raw dataset is assessed for completeness and consistency by examining missing values and duplicate records before transformation.

In [5]:
# ============================================================
# 5. Initial Data Quality Assessment
# ============================================================

total_missing = raw_df.isna().sum().sum()
duplicate_rows = raw_df.duplicated().sum()

print(f"Total missing values: {total_missing}")
print(f"Duplicate rows: {duplicate_rows}")

missing_by_column = raw_df.isna().sum().sort_values(ascending=False)
missing_by_column.head(10)

Total missing values: 54265
Duplicate rows: 0


2025    1401
1960    1266
1963    1233
1961    1233
1962    1230
1964    1223
1966    1219
1965    1219
1967    1188
1968    1184
dtype: int64

## Explore Dataset Scope

The overall characteristics of the raw dataset are explored to determine the number of indicators and the temporal coverage available for analysis.

In [6]:
# ============================================================
# 6. Explore Dataset Scope
# ============================================================

num_indicators = raw_df["Indicator Name"].nunique()

print(f"Unique indicators : {num_indicators}")
print(f"Dataset shape : {raw_df.shape}")
print(f"Year coverage : {min(year_columns)} - {max(year_columns)}")

Unique indicators : 1486
Dataset shape : (1486, 70)
Year coverage : 1960 - 2025


## Phase 2 – Data Transformation

This phase transforms the raw World Bank dataset into a focused analytical dataset. The transformation process begins by selecting indicators aligned with the project objective, validating their availability, filtering the raw dataset, reshaping the data from wide to long format, and preparing it for feature engineering.

## Indicator Selection

The selected indicators are aligned with a citizen-centred financial well-being perspective. The framework combines macroeconomic conditions, household financial behaviour and financial system support indicators to show how Malaysia's financial performance relates to the daily financial experience of ordinary Malaysians.

In [7]:
# ============================================================
# 7. Define Selected Indicators
# ============================================================

indicator_catalog = pd.DataFrame({
    "Indicator_Code": [
        "NY.GDP.MKTP.KD.ZG",
        "FP.CPI.TOTL.ZG",
        "SL.UEM.TOTL.ZS",
        "NY.ADJ.NNTY.PC.KD",
        "NE.CON.PRVT.PC.KD",
        "NY.GDS.TOTL.ZS",
        "FR.INR.LEND",
        "FR.INR.DPST",
        "FD.AST.PRVT.GD.ZS",
        "FB.AST.NPER.ZS"
    ],
    "Indicator": [
        "GDP growth (annual %)",
        "Inflation, consumer prices (annual %)",
        "Unemployment, total (% of total labor force) (modeled ILO estimate)",
        "Adjusted net national income per capita (constant 2015 US$)",
        "Households and NPISHs Final consumption expenditure per capita (constant 2015 US$)",
        "Gross domestic savings (% of GDP)",
        "Lending interest rate (%)",
        "Deposit interest rate (%)",
        "Domestic credit to private sector by banks (% of GDP)",
        "Bank nonperforming loans to total gross loans (%)"
    ],
    "Short_Name": [
        "GDP Growth",
        "Inflation",
        "Unemployment",
        "Income per Capita",
        "Household Consumption",
        "Gross Domestic Savings",
        "Lending Rate",
        "Deposit Rate",
        "Domestic Credit",
        "Non-performing Loans"
    ],
    "Theme": [
        "Economic Well-being",
        "Economic Well-being",
        "Economic Well-being",
        "Economic Well-being",
        "Household Financial Behaviour",
        "Household Financial Behaviour",
        "Financial System Support",
        "Financial System Support",
        "Financial System Support",
        "Financial System Support"
    ]
})

selected_indicators = indicator_catalog["Indicator"].tolist()
selected_indicator_codes = indicator_catalog["Indicator_Code"].tolist()

indicator_catalog

,Indicator_Code,Indicator,Short_Name,Theme
0,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GDP Growth,Economic Well-being
1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",Inflation,Economic Well-being
2,SL.UEM.TOTL.ZS,"Unemployment, total (% of total labor force) (...",Unemployment,Economic Well-being
3,NY.ADJ.NNTY.PC.KD,Adjusted net national income per capita (const...,Income per Capita,Economic Well-being
4,NE.CON.PRVT.PC.KD,Households and NPISHs Final consumption expend...,Household Consumption,Household Financial Behaviour
5,NY.GDS.TOTL.ZS,Gross domestic savings (% of GDP),Gross Domestic Savings,Household Financial Behaviour
6,FR.INR.LEND,Lending interest rate (%),Lending Rate,Financial System Support
7,FR.INR.DPST,Deposit interest rate (%),Deposit Rate,Financial System Support
8,FD.AST.PRVT.GD.ZS,Domestic credit to private sector by banks (% ...,Domestic Credit,Financial System Support
9,FB.AST.NPER.ZS,Bank nonperforming loans to total gross loans (%),Non-performing Loans,Financial System Support


In [8]:
# ============================================================
# 8. Validate Selected Indicators
# ============================================================

available_indicator_codes = set(raw_df["Indicator Code"].dropna())

indicator_validation = indicator_catalog.copy()
indicator_validation["Available"] = indicator_validation["Indicator_Code"].apply(
    lambda x: x in available_indicator_codes
)

indicator_validation

,Indicator_Code,Indicator,Short_Name,Theme,Available
0,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GDP Growth,Economic Well-being,True
1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",Inflation,Economic Well-being,True
2,SL.UEM.TOTL.ZS,"Unemployment, total (% of total labor force) (...",Unemployment,Economic Well-being,True
3,NY.ADJ.NNTY.PC.KD,Adjusted net national income per capita (const...,Income per Capita,Economic Well-being,True
4,NE.CON.PRVT.PC.KD,Households and NPISHs Final consumption expend...,Household Consumption,Household Financial Behaviour,True
5,NY.GDS.TOTL.ZS,Gross domestic savings (% of GDP),Gross Domestic Savings,Household Financial Behaviour,True
6,FR.INR.LEND,Lending interest rate (%),Lending Rate,Financial System Support,True
7,FR.INR.DPST,Deposit interest rate (%),Deposit Rate,Financial System Support,True
8,FD.AST.PRVT.GD.ZS,Domestic credit to private sector by banks (% ...,Domestic Credit,Financial System Support,True
9,FB.AST.NPER.ZS,Bank nonperforming loans to total gross loans (%),Non-performing Loans,Financial System Support,True


In [9]:
# ============================================================
# 9. Check Indicator Availability
# ============================================================

available_indicator_codes = raw_df["Indicator Code"].dropna().unique()

indicator_catalog["Available"] = indicator_catalog["Indicator_Code"].isin(available_indicator_codes)

indicator_catalog

,Indicator_Code,Indicator,Short_Name,Theme,Available
0,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GDP Growth,Economic Well-being,True
1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",Inflation,Economic Well-being,True
2,SL.UEM.TOTL.ZS,"Unemployment, total (% of total labor force) (...",Unemployment,Economic Well-being,True
3,NY.ADJ.NNTY.PC.KD,Adjusted net national income per capita (const...,Income per Capita,Economic Well-being,True
4,NE.CON.PRVT.PC.KD,Households and NPISHs Final consumption expend...,Household Consumption,Household Financial Behaviour,True
5,NY.GDS.TOTL.ZS,Gross domestic savings (% of GDP),Gross Domestic Savings,Household Financial Behaviour,True
6,FR.INR.LEND,Lending interest rate (%),Lending Rate,Financial System Support,True
7,FR.INR.DPST,Deposit interest rate (%),Deposit Rate,Financial System Support,True
8,FD.AST.PRVT.GD.ZS,Domestic credit to private sector by banks (% ...,Domestic Credit,Financial System Support,True
9,FB.AST.NPER.ZS,Bank nonperforming loans to total gross loans (%),Non-performing Loans,Financial System Support,True


In [10]:
# ============================================================
# 10. Filter Selected Indicators
# ============================================================

selected_df = raw_df[
    raw_df["Indicator Code"].isin(selected_indicator_codes)
].copy()

selected_df = selected_df.merge(
    indicator_catalog[["Indicator_Code", "Theme", "Short_Name"]],
    left_on="Indicator Code",
    right_on="Indicator_Code",
    how="left"
)

selected_df = selected_df.drop(columns=["Indicator_Code"])
selected_df = selected_df.rename(columns={"Short_Name": "Indicator"})

print("Selected indicator rows:", selected_df.shape[0])
print("Selected indicators found:", selected_df["Indicator"].nunique())
print("Missing themes after merge:", selected_df["Theme"].isna().sum())
print("Missing short names after merge:", selected_df["Indicator"].isna().sum())

selected_df[["Indicator Name", "Indicator Code", "Theme", "Indicator"]]

Selected indicator rows: 10
Selected indicators found: 10
Missing themes after merge: 0
Missing short names after merge: 0


,Indicator Name,Indicator Code,Theme,Indicator
0,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,Household Financial Behaviour,Household Consumption
1,Lending interest rate (%),FR.INR.LEND,Financial System Support,Lending Rate
2,Domestic credit to private sector by banks (% ...,FD.AST.PRVT.GD.ZS,Financial System Support,Domestic Credit
3,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,Economic Well-being,Unemployment
4,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,Economic Well-being,Inflation
5,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate
6,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans
7,Gross domestic savings (% of GDP),NY.GDS.TOTL.ZS,Household Financial Behaviour,Gross Domestic Savings
8,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth
9,Adjusted net national income per capita (const...,NY.ADJ.NNTY.PC.KD,Economic Well-being,Income per Capita


## Reshape Dataset

The selected dataset is currently in wide format, where each year is stored as a separate column. For analysis and visualisation, the dataset is converted into long format so that each row represents one indicator-year observation.

In [11]:
# ============================================================
# 11. Reshape Dataset from Wide to Long Format
# ============================================================

year_columns = [col for col in selected_df.columns if str(col).isdigit()]

analysis_df = selected_df.melt(
    id_vars=[
        "Country Name",
        "Country Code",
        "Indicator Name",
        "Indicator Code",
        "Theme",
        "Indicator"
    ],
    value_vars=year_columns,
    var_name="Year",
    value_name="Value"
)

analysis_df["Year"] = analysis_df["Year"].astype(int)
analysis_df["Value"] = pd.to_numeric(analysis_df["Value"], errors="coerce")

analysis_df = analysis_df.sort_values(
    by=["Theme", "Indicator", "Year"]
).reset_index(drop=True)

print("Long-format dataset created.")
print(f"Rows: {analysis_df.shape[0]}")
print(f"Columns: {analysis_df.shape[1]}")

analysis_df.head()

Long-format dataset created.
Rows: 660
Columns: 8


,Country Name,Country Code,Indicator Name,Indicator Code,Theme,Indicator,Year,Value
0,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,1960,NaN
1,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,1961,7.597994
2,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,1962,6.421030
3,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,1963,7.338804
4,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,1964,5.358963


## Analysis Period Selection

The analytical period is fixed at **2000–2024**. This period provides a long-term view of Malaysia's financial and economic conditions while excluding incomplete future-year observations. Some indicators may have shorter available periods due to source publication limitations, such as non-performing loans and income per capita.

In [12]:
# ============================================================
# 12. Select Analysis Period
# ============================================================

analysis_df = analysis_df[
    (analysis_df["Year"] >= 2000) &
    (analysis_df["Year"] <= 2024)
].copy()

print(f"Initial years covered: {analysis_df['Year'].min()} - {analysis_df['Year'].max()}")
print("Indicators retained:", analysis_df["Indicator"].nunique())

analysis_df.head()

Initial years covered: 2000 - 2024
Indicators retained: 10


,Country Name,Country Code,Indicator Name,Indicator Code,Theme,Indicator,Year,Value
40,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,2000,8.858868
41,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,2001,0.517675
42,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,2002,5.390988
43,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,2003,5.788499
44,Malaysia,MYS,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,Economic Well-being,GDP Growth,2004,6.783438


## Data Quality Assessment

Following the selection of the analytical period, the transformed dataset is assessed to identify missing values, duplicate records and data type inconsistencies. This assessment ensures that only relevant observations are cleaned before feature engineering and export.

In [13]:
# ============================================================
# 13. Data Quality Assessment
# ============================================================

print("Dataset Summary")
print("-" * 40)

print(f"Rows                : {analysis_df.shape[0]}")
print(f"Columns             : {analysis_df.shape[1]}")
print(f"Missing values      : {analysis_df.isna().sum().sum()}")
print(f"Duplicate rows      : {analysis_df.duplicated().sum()}")

print("\nData Types")
print("-" * 40)
print(analysis_df.dtypes)

Dataset Summary
----------------------------------------
Rows                : 250
Columns             : 8
Missing values      : 9
Duplicate rows      : 0

Data Types
----------------------------------------
Country Name          str
Country Code          str
Indicator Name        str
Indicator Code        str
Theme                 str
Indicator             str
Year                int64
Value             float64
dtype: object


In [14]:
# ============================================================
# 14. Missing Values by Indicator
# ============================================================

missing_summary = (
    analysis_df
    .groupby("Indicator Name")["Value"]
    .apply(lambda x: x.isna().sum())
    .reset_index(name="Missing Values")
)

missing_summary

,Indicator Name,Missing Values
0,Adjusted net national income per capita (const...,3
1,Bank nonperforming loans to total gross loans (%),6
2,Deposit interest rate (%),0
3,Domestic credit to private sector by banks (% ...,0
4,GDP growth (annual %),0
5,Gross domestic savings (% of GDP),0
6,Households and NPISHs Final consumption expend...,0
7,"Inflation, consumer prices (annual %)",0
8,Lending interest rate (%),0
9,"Unemployment, total (% of total labor force) (...",0


## Missing Value Investigation

Before applying any data cleaning technique, the locations of missing values are investigated to determine whether they represent data quality issues or unavailable observations from the data source.

In [15]:
# ============================================================
# 15. Locate Missing Values
# ============================================================

missing_records = analysis_df[
    analysis_df["Value"].isna()
].sort_values(["Indicator Name", "Year"])

missing_records

,Country Name,Country Code,Indicator Name,Indicator Code,Theme,Indicator,Year,Value
128,Malaysia,MYS,Adjusted net national income per capita (const...,NY.ADJ.NNTY.PC.KD,Economic Well-being,Income per Capita,2022,NaN
129,Malaysia,MYS,Adjusted net national income per capita (const...,NY.ADJ.NNTY.PC.KD,Economic Well-being,Income per Capita,2023,NaN
130,Malaysia,MYS,Adjusted net national income per capita (const...,NY.ADJ.NNTY.PC.KD,Economic Well-being,Income per Capita,2024,NaN
502,Malaysia,MYS,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans,2000,NaN
503,Malaysia,MYS,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans,2001,NaN
504,Malaysia,MYS,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans,2002,NaN
505,Malaysia,MYS,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans,2003,NaN
506,Malaysia,MYS,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans,2004,NaN
526,Malaysia,MYS,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Financial System Support,Non-performing Loans,2024,NaN


## Analytical Dataset Preparation

The missing values identified in the transformed dataset were investigated before cleaning. These missing observations reflect unavailable official statistics for specific indicator-year combinations rather than data entry errors. Missing values are removed before export so that the final analytical datasets are ready for R visualisation and dashboard development.

In [16]:
# ============================================================
# 16. Finalise Analytical Dataset
# ============================================================

analysis_df = analysis_df.copy()

print(f"Rows retained : {analysis_df.shape[0]}")
print(f"Years covered : {analysis_df['Year'].min()} - {analysis_df['Year'].max()}")

Rows retained : 250
Years covered : 2000 - 2024


In [17]:
# ============================================================
# 17. Remove Remaining Missing Values
# ============================================================

analysis_df = analysis_df.dropna(subset=["Value"]).reset_index(drop=True)

print(f"Rows after cleaning : {analysis_df.shape[0]}")
print(f"Missing values remaining : {analysis_df['Value'].isna().sum()}")

Rows after cleaning : 241
Missing values remaining : 0


## Feature Engineering and Validation

Theme and short indicator names are assigned from the indicator catalog. The pipeline avoids manual remapping after the reshape process to prevent old or inconsistent indicator labels from reappearing. Final validation checks confirm that the selected indicators, themes and available years are internally consistent before export.

In [18]:
# ============================================================
# 18. Validate Indicator Themes
# ============================================================

# Indicator names and themes were assigned from indicator_catalog in Section 10.
# Do not recreate mappings here.

theme_check = (
    analysis_df[["Indicator Name", "Indicator Code", "Indicator", "Theme"]]
    .drop_duplicates()
    .sort_values(["Theme", "Indicator"])
    .reset_index(drop=True)
)

theme_check

,Indicator Name,Indicator Code,Indicator,Theme
0,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,GDP Growth,Economic Well-being
1,Adjusted net national income per capita (const...,NY.ADJ.NNTY.PC.KD,Income per Capita,Economic Well-being
2,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,Inflation,Economic Well-being
3,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,Unemployment,Economic Well-being
4,Deposit interest rate (%),FR.INR.DPST,Deposit Rate,Financial System Support
5,Domestic credit to private sector by banks (% ...,FD.AST.PRVT.GD.ZS,Domestic Credit,Financial System Support
6,Lending interest rate (%),FR.INR.LEND,Lending Rate,Financial System Support
7,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Non-performing Loans,Financial System Support
8,Gross domestic savings (% of GDP),NY.GDS.TOTL.ZS,Gross Domestic Savings,Household Financial Behaviour
9,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,Household Consumption,Household Financial Behaviour


In [19]:
# ============================================================
# 19. Validate Final Indicator Names
# ============================================================

indicator_name_check = (
    analysis_df[["Indicator Name", "Indicator Code", "Indicator", "Theme"]]
    .drop_duplicates()
    .sort_values(["Theme", "Indicator"])
    .reset_index(drop=True)
)

print("Missing Indicator labels:", analysis_df["Indicator"].isna().sum())
print("Missing Theme labels:", analysis_df["Theme"].isna().sum())

indicator_name_check

Missing Indicator labels: 0
Missing Theme labels: 0


,Indicator Name,Indicator Code,Indicator,Theme
0,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,GDP Growth,Economic Well-being
1,Adjusted net national income per capita (const...,NY.ADJ.NNTY.PC.KD,Income per Capita,Economic Well-being
2,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,Inflation,Economic Well-being
3,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,Unemployment,Economic Well-being
4,Deposit interest rate (%),FR.INR.DPST,Deposit Rate,Financial System Support
5,Domestic credit to private sector by banks (% ...,FD.AST.PRVT.GD.ZS,Domestic Credit,Financial System Support
6,Lending interest rate (%),FR.INR.LEND,Lending Rate,Financial System Support
7,Bank nonperforming loans to total gross loans (%),FB.AST.NPER.ZS,Non-performing Loans,Financial System Support
8,Gross domestic savings (% of GDP),NY.GDS.TOTL.ZS,Gross Domestic Savings,Household Financial Behaviour
9,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,Household Consumption,Household Financial Behaviour


## Analytical Dataset Validation

Before exporting the analytical dataset, a final validation is performed to verify that the transformed data are complete, internally consistent and ready for downstream visual analytics. The validation confirms that all selected indicators, study years, analytical phases and thematic groupings have been correctly generated.

In [20]:
# ============================================================
# 20. Analytical Dataset Overview
# ============================================================

print("Analytical Dataset Summary")
print("-" * 40)

print(f"Rows                : {analysis_df.shape[0]}")
print(f"Columns             : {analysis_df.shape[1]}")
print(f"Years               : {analysis_df['Year'].min()} - {analysis_df['Year'].max()}")
print(f"Indicators          : {analysis_df['Indicator'].nunique()}")
print(f"Themes              : {analysis_df['Theme'].nunique()}")
print(f"Missing Values      : {analysis_df.isna().sum().sum()}")
print(f"Duplicate Rows      : {analysis_df.duplicated().sum()}")

Analytical Dataset Summary
----------------------------------------
Rows                : 241
Columns             : 8
Years               : 2000 - 2024
Indicators          : 10
Themes              : 3
Missing Values      : 0
Duplicate Rows      : 0


In [21]:
analysis_df.columns.tolist()

['Country Name',
 'Country Code',
 'Indicator Name',
 'Indicator Code',
 'Theme',
 'Indicator',
 'Year',
 'Value']

## Analytical Dataset Validation

Before exporting the analytical dataset, a final validation is performed to verify that the transformed data are complete, internally consistent and ready for downstream visual analytics.

In [22]:
# ============================================================
# 21. Validate Indicator Distribution
# ============================================================

indicator_summary = (
    analysis_df
    .groupby("Indicator")
    .agg(
        Observations=("Value", "count"),
        Start_Year=("Year", "min"),
        End_Year=("Year", "max")
    )
    .reset_index()
)

indicator_summary

,Indicator,Observations,Start_Year,End_Year
0,Deposit Rate,25,2000,2024
1,Domestic Credit,25,2000,2024
2,GDP Growth,25,2000,2024
3,Gross Domestic Savings,25,2000,2024
4,Household Consumption,25,2000,2024
5,Income per Capita,22,2000,2021
6,Inflation,25,2000,2024
7,Lending Rate,25,2000,2024
8,Non-performing Loans,19,2005,2023
9,Unemployment,25,2000,2024


In [23]:
# ============================================================
# 22. Validate Indicator Themes
# ============================================================

theme_summary = (
    analysis_df
    .groupby("Theme")
    .agg(
        Indicators=("Indicator", "nunique"),
        Observations=("Value", "count")
    )
    .reset_index()
)

theme_summary

,Theme,Indicators,Observations
0,Economic Well-being,4,97
1,Financial System Support,4,94
2,Household Financial Behaviour,2,50


In [24]:
# ============================================================
# 23. Preview Final Analytical Dataset
# ============================================================

analysis_df = analysis_df.sort_values(
    ["Indicator", "Year"]
).reset_index(drop=True)

analysis_df.head(15)

,Country Name,Country Code,Indicator Name,Indicator Code,Theme,Indicator,Year,Value
0,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2000,3.362500
1,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2001,3.374167
2,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2002,3.205000
3,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2003,3.066667
4,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2004,3.000000
5,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2005,3.001667
6,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2006,3.149167
7,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2007,3.165833
8,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2008,3.125833
9,Malaysia,MYS,Deposit interest rate (%),FR.INR.DPST,Financial System Support,Deposit Rate,2009,2.081667


## PHASE 3 – DATA DELIVERY

## Export Analytical Datasets

The validated analytical dataset is exported into multiple formats to support different downstream analytical tasks. Each dataset is designed for a specific purpose within the Malaysia Financial Analytics Platform.

In [25]:
# ============================================================
# Export Output Folders
# ============================================================

FINAL_DIR = DATA_FINAL
METADATA_DIR = BASE_DIR / "data" / "metadata"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Output folders ready.")

Output folders ready.


In [26]:
# ============================================================
# Export Full Analytical Dataset
# ============================================================

analysis_df.to_csv(
    FINAL_DIR / "analysis_dataset_full.csv",
    index=False
)

print("analysis_dataset_full.csv exported successfully.")

analysis_dataset_full.csv exported successfully.


In [27]:
# ============================================================
# Create Dashboard Dataset
# ============================================================

dashboard_df = analysis_df[
    [
        "Indicator",
        "Theme",
        "Year",
        "Value"
    ]
].copy()

dashboard_df.to_csv(
    FINAL_DIR / "analysis_dataset_dashboard.csv",
    index=False
)

print("analysis_dataset_dashboard.csv exported successfully.")

analysis_dataset_dashboard.csv exported successfully.


In [28]:
# ============================================================
# Create Heatmap Dataset
# ============================================================

heatmap_df = dashboard_df.pivot(
    index="Year",
    columns="Indicator",
    values="Value"
).reset_index()

heatmap_df.to_csv(
    FINAL_DIR / "analysis_dataset_heatmap.csv",
    index=False
)

print("analysis_dataset_heatmap.csv exported successfully.")

analysis_dataset_heatmap.csv exported successfully.


In [29]:
# ============================================================
# Export Metadata
# ============================================================

indicator_catalog.to_csv(
    METADATA_DIR / "indicator_catalog.csv",
    index=False
)

print("indicator_catalog.csv exported successfully.")

indicator_catalog.csv exported successfully.


## Executive KPI Summary

In addition to the analytical datasets, an executive summary of key performance indicators (KPIs) is generated. This summary provides descriptive statistics and period comparisons for each selected indicator. The KPI summary supports executive reporting, dashboard development and project documentation.

In [30]:
# ============================================================
# Generate Executive KPI Summary
# ============================================================

kpi_summary = (
    analysis_df
    .groupby(["Indicator", "Theme"])
    .agg(
        Mean=("Value", "mean"),
        Minimum=("Value", "min"),
        Maximum=("Value", "max"),
        Std_Dev=("Value", "std"),
        Start_Year=("Year", "min"),
        End_Year=("Year", "max"),
        Observations=("Value", "count")
    )
    .reset_index()
)

kpi_summary

,Indicator,Theme,Mean,Minimum,Maximum,Std_Dev,Start_Year,End_Year,Observations
0,Deposit Rate,Financial System Support,2.836877,1.561721,3.374167,0.473374,2000,2024,25
1,Domestic Credit,Financial System Support,116.193252,96.604568,133.785749,8.923769,2000,2024,25
2,GDP Growth,Economic Well-being,4.676369,-5.456847,9.031737,3.028618,2000,2024,25
3,Gross Domestic Savings,Household Financial Behaviour,36.453398,26.025698,46.080398,6.401713,2000,2024,25
4,Household Consumption,Household Financial Behaviour,4647.795831,2635.634458,7208.161951,1463.029789,2000,2024,25
5,Income per Capita,Economic Well-being,6089.955818,3887.196756,7863.871552,1206.895715,2000,2021,22
6,Inflation,Economic Well-being,2.090753,-1.138702,5.440782,1.302070,2000,2024,25
7,Lending Rate,Financial System Support,5.321223,3.444106,7.673333,1.047283,2000,2024,25
8,Non-performing Loans,Financial System Support,3.091274,1.468191,9.390240,2.459841,2005,2023,19
9,Unemployment,Economic Well-being,3.484160,2.880000,4.640000,0.427357,2000,2024,25


In [31]:
# ============================================================
# Latest Available Value
# ============================================================

latest_values = (
    analysis_df
    .sort_values("Year")
    .groupby("Indicator")
    .tail(1)[["Indicator", "Year", "Value"]]
    .rename(columns={
        "Year": "Latest_Year",
        "Value": "Latest_Value"
    })
)

latest_values

,Indicator,Latest_Year,Latest_Value
146,Income per Capita,2021,7437.761732
215,Non-performing Loans,2023,1.653156
124,Household Consumption,2024,7208.161951
74,GDP Growth,2024,5.105459
49,Domestic Credit,2024,116.083822
24,Deposit Rate,2024,2.647115
99,Gross Domestic Savings,2024,27.273482
171,Inflation,2024,1.834100
196,Lending Rate,2024,5.282681
240,Unemployment,2024,3.846000


In [32]:
# ============================================================
# Percentage Change
# ============================================================

first_values = (
    analysis_df
    .sort_values("Year")
    .groupby("Indicator")
    .head(1)[["Indicator", "Year", "Value"]]
    .rename(columns={
        "Year": "First_Year",
        "Value": "First_Value"
    })
)

latest_values_for_change = (
    analysis_df
    .sort_values("Year")
    .groupby("Indicator")
    .tail(1)[["Indicator", "Year", "Value"]]
    .rename(columns={
        "Year": "Latest_Year_For_Change",
        "Value": "Latest_Value_For_Change"
    })
)

change_df = first_values.merge(
    latest_values_for_change,
    on="Indicator",
    how="left"
)

change_df["Percent_Change"] = (
    (change_df["Latest_Value_For_Change"] - change_df["First_Value"])
    / change_df["First_Value"]
) * 100

change_df

,Indicator,First_Year,First_Value,Latest_Year_For_Change,Latest_Value_For_Change,Percent_Change
0,Deposit Rate,2000,3.362500,2024,2.647115,-21.275377
1,Domestic Credit,2000,126.729338,2024,116.083822,-8.400199
2,GDP Growth,2000,8.858868,2024,5.105459,-42.368945
3,Gross Domestic Savings,2000,46.080398,2024,27.273482,-40.813268
4,Household Consumption,2000,2635.634458,2024,7208.161951,173.488682
5,Income per Capita,2000,4222.304132,2021,7437.761732,76.154097
6,Lending Rate,2000,7.673333,2024,5.282681,-31.155330
7,Inflation,2000,1.534740,2024,1.834100,19.505579
8,Unemployment,2000,3.000000,2024,3.846000,28.200000
9,Non-performing Loans,2005,9.390240,2023,1.653156,-82.394956


In [33]:
# ============================================================
# Create Executive KPI Summary
# ============================================================

executive_kpi = (
    kpi_summary
    .merge(latest_values, on="Indicator", how="left")
    .merge(
        change_df[
            [
                "Indicator",
                "First_Year",
                "First_Value",
                "Latest_Year_For_Change",
                "Latest_Value_For_Change",
                "Percent_Change"
            ]
        ],
        on="Indicator",
        how="left"
    )
)

executive_kpi = executive_kpi.round(2)

executive_kpi

,Indicator,Theme,Mean,Minimum,Maximum,Std_Dev,Start_Year,End_Year,Observations,Latest_Year,Latest_Value,First_Year,First_Value,Latest_Year_For_Change,Latest_Value_For_Change,Percent_Change
0,Deposit Rate,Financial System Support,2.84,1.56,3.37,0.47,2000,2024,25,2024,2.65,2000,3.36,2024,2.65,-21.28
1,Domestic Credit,Financial System Support,116.19,96.60,133.79,8.92,2000,2024,25,2024,116.08,2000,126.73,2024,116.08,-8.40
2,GDP Growth,Economic Well-being,4.68,-5.46,9.03,3.03,2000,2024,25,2024,5.11,2000,8.86,2024,5.11,-42.37
3,Gross Domestic Savings,Household Financial Behaviour,36.45,26.03,46.08,6.40,2000,2024,25,2024,27.27,2000,46.08,2024,27.27,-40.81
4,Household Consumption,Household Financial Behaviour,4647.80,2635.63,7208.16,1463.03,2000,2024,25,2024,7208.16,2000,2635.63,2024,7208.16,173.49
5,Income per Capita,Economic Well-being,6089.96,3887.20,7863.87,1206.90,2000,2021,22,2021,7437.76,2000,4222.30,2021,7437.76,76.15
6,Inflation,Economic Well-being,2.09,-1.14,5.44,1.30,2000,2024,25,2024,1.83,2000,1.53,2024,1.83,19.51
7,Lending Rate,Financial System Support,5.32,3.44,7.67,1.05,2000,2024,25,2024,5.28,2000,7.67,2024,5.28,-31.16
8,Non-performing Loans,Financial System Support,3.09,1.47,9.39,2.46,2005,2023,19,2023,1.65,2005,9.39,2023,1.65,-82.39
9,Unemployment,Economic Well-being,3.48,2.88,4.64,0.43,2000,2024,25,2024,3.85,2000,3.00,2024,3.85,28.20


In [34]:
# ============================================================
# Export Executive KPI Summary
# ============================================================

executive_kpi.to_csv(
    FINAL_DIR / "executive_kpi_summary.csv",
    index=False
)

print("executive_kpi_summary.csv exported successfully.")

executive_kpi_summary.csv exported successfully.


## Project Metadata

To improve reusability across different analytical components, a project metadata file is generated. The metadata records the project scope, analytical period, selected indicators and output configuration, allowing downstream applications such as the R visualisation notebook and Shiny dashboard to automatically retrieve project settings without hard-coded values.

In [35]:
# ============================================================
# Create Project Metadata
# ============================================================

project_metadata = pd.DataFrame({
    "Parameter": [
        "Project Name",
        "Country",
        "Data Source",
        "Study Period",
        "Number of Indicators",
        "Number of Themes",
        "Dashboard Platform",
        "Visualisation Platform",
        "Analytical Dataset",
        "Project Focus"
    ],
    "Value": [
        "Malaysia Financial Performance Analytics Platform",
        "Malaysia",
        "World Bank Open Data",
        "2000-2024",
        analysis_df["Indicator"].nunique(),
        analysis_df["Theme"].nunique(),
        "R Shiny",
        "R / ggplot2 / plotly",
        "analysis_dataset_dashboard.csv",
        "Citizen-centred financial well-being and financial system support"
    ]
})

project_metadata

,Parameter,Value
0,Project Name,Malaysia Financial Performance Analytics Platform
1,Country,Malaysia
2,Data Source,World Bank Open Data
3,Study Period,2000-2024
4,Number of Indicators,10
5,Number of Themes,3
6,Dashboard Platform,R Shiny
7,Visualisation Platform,R / ggplot2 / plotly
8,Analytical Dataset,analysis_dataset_dashboard.csv
9,Project Focus,Citizen-centred financial well-being and finan...


In [36]:
# ============================================================
# Export Datasets and Metadata
# ============================================================

analysis_df.to_csv(DATA_FINAL / "analysis_dataset_full.csv", index=False)
dashboard_df.to_csv(DATA_FINAL / "analysis_dataset_dashboard.csv", index=False)
heatmap_df.to_csv(DATA_FINAL / "analysis_dataset_heatmap.csv", index=False)
executive_kpi.to_csv(DATA_FINAL / "executive_kpi_summary.csv", index=False)

indicator_catalog.to_csv(DATA_METADATA / "indicator_catalog.csv", index=False)
project_metadata.to_csv(DATA_METADATA / "project_metadata.csv", index=False)
indicator_summary.to_csv(DATA_METADATA / "indicator_summary.csv", index=False)
theme_summary.to_csv(DATA_METADATA / "theme_summary.csv", index=False)

print("All datasets exported successfully.")

All datasets exported successfully.


## Pipeline Completion Summary

The Malaysia Financial Analytics Platform ETL pipeline has successfully transformed the raw World Bank Malaysia dataset into a structured analytical dataset suitable for downstream analytics.

The generated outputs include analytical datasets, metadata and executive KPI summaries that will be consumed by the R visual analytics workflow, interactive dashboard and executive reporting components.

In [37]:
# ============================================================
# ETL Pipeline Summary
# ============================================================

print("=" * 65)
print(" MALAYSIA FINANCIAL PERFORMANCE ANALYTICS PLATFORM ")
print(" CITIZEN-CENTRED FINANCIAL PERFORMANCE ETL PIPELINE COMPLETED SUCCESSFULLY ")
print("=" * 65)

print("\nINPUT")
print("-" * 40)
print("Source                : World Bank Open Data")
print("Country               : Malaysia")
print(f"Raw Indicators        : {raw_df['Indicator Name'].nunique():,}")
print("Original Years        : 1960–2025")

print("\nOUTPUT")
print("-" * 40)
print(f"Study Period          : {analysis_df['Year'].min()}–{analysis_df['Year'].max()}")
print(f"Selected Indicators   : {analysis_df['Indicator'].nunique()}")
print(f"Themes                : {analysis_df['Theme'].nunique()}")
print(f"Final Records         : {analysis_df.shape[0]}")

print("\nTHEMES")
print("-" * 40)
for theme in sorted(analysis_df['Theme'].unique()):
    print(f"✓ {theme}")

print("\nGENERATED FILES")
print("-" * 40)
print("✓ analysis_dataset_full.csv")
print("✓ analysis_dataset_dashboard.csv")
print("✓ analysis_dataset_heatmap.csv")
print("✓ indicator_catalog.csv")
print("✓ executive_kpi_summary.csv")
print("✓ project_metadata.csv")

print("\nREADY FOR")
print("-" * 40)
print("✓ R Visual Analytics Notebook")
print("✓ Interactive Shiny Dashboard")
print("✓ Heatmap & Correlation Analysis")
print("✓ Executive Storyboard")
print("✓ Project README Documentation")

print("\nETL Pipeline completed successfully.")

 MALAYSIA FINANCIAL PERFORMANCE ANALYTICS PLATFORM 
 CITIZEN-CENTRED FINANCIAL PERFORMANCE ETL PIPELINE COMPLETED SUCCESSFULLY 

INPUT
----------------------------------------
Source                : World Bank Open Data
Country               : Malaysia
Raw Indicators        : 1,486
Original Years        : 1960–2025

OUTPUT
----------------------------------------
Study Period          : 2000–2024
Selected Indicators   : 10
Themes                : 3
Final Records         : 241

THEMES
----------------------------------------
✓ Economic Well-being
✓ Financial System Support
✓ Household Financial Behaviour

GENERATED FILES
----------------------------------------
✓ analysis_dataset_full.csv
✓ analysis_dataset_dashboard.csv
✓ analysis_dataset_heatmap.csv
✓ indicator_catalog.csv
✓ executive_kpi_summary.csv
✓ project_metadata.csv

READY FOR
----------------------------------------
✓ R Visual Analytics Notebook
✓ Interactive Shiny Dashboard
✓ Heatmap & Correlation Analysis
✓ Executive Storyb

In [38]:
dashboard_df.head()
dashboard_df.columns.tolist()
indicator_summary
project_metadata

,Parameter,Value
0,Project Name,Malaysia Financial Performance Analytics Platform
1,Country,Malaysia
2,Data Source,World Bank Open Data
3,Study Period,2000-2024
4,Number of Indicators,10
5,Number of Themes,3
6,Dashboard Platform,R Shiny
7,Visualisation Platform,R / ggplot2 / plotly
8,Analytical Dataset,analysis_dataset_dashboard.csv
9,Project Focus,Citizen-centred financial well-being and finan...
